**Week 11 — Logistic Regression ธุรกิจ — z, odds ratio, ความน่าจะเป็น, confusion matrix, threshold, class_weight**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:** `cust = pd.read_csv('biz_customers.csv')` (1203 แถว แถวละ 1 คน)
- **การแบ่งชุดข้อมูล:** train = 960 แถวแรก (`.iloc[:960]`), test = 243 แถวที่เหลือ (`.iloc[960:]`) โดยไม่สลับข้อมูล
- **แบบจำลอง Logistic Regression:** ประมาณค่าด้วย Maximum Likelihood Estimation (ไม่มี Regularization) ผ่าน `statsmodels.formula.api.smf.logit` หรือ `LogisticRegression(penalty=None)` ใน scikit-learn
- **การตัดสินใจคลาส:** ทำนายเป็นคลาส 1 เมื่อความน่าจะเป็นที่ทำนาย $\hat{p} \ge \text{threshold}$ (คลาสบวก = 1)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์

### รายละเอียดไฟล์ข้อมูล `biz_customers.csv` (1203 แถว)

- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `churn`: ลูกค้าเลิกซื้อหรือไม่ (1 = เลิกซื้อ, 0 = ยังซื้ออยู่)
- `responded`: ลูกค้าตอบรับโปรโมชันหรือไม่ (1 = ตอบรับ, 0 = ไม่ตอบรับ)

**โจทย์ข้อ 11.1:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `income_k`, `age` ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ประมาณด้วย Maximum Likelihood (ไม่มี regularization) ค่าสถิติ $z$ (สัมประสิทธิ์ $\div$ standard error) ของ `email_opens` = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้าทั้งหมด
cust = pd.read_csv('biz_customers.csv')

# 2. Fit Logistic Regression ด้วย MLE
formula = 'responded ~ email_opens + income_k + age'
model = smf.logit(formula, data=cust).fit()
print(model.summary())

# 3. ค่าสถิติ z ของ email_opens
z_email = model.tvalues['email_opens']
print(f"z-statistic ของ email_opens: {z_email:.4f}")
print(f"คำตอบข้อ 11.1: {z_email:.4f}")


Optimization terminated successfully.
         Current function value: 0.588781
         Iterations 5
z-statistic ของ email_opens: 6.2677
คำตอบข้อ 11.1: 6.2677


**โจทย์ข้อ 11.2:** กรณีศึกษาทีมวิเคราะห์ของแพลตฟอร์มอีคอมเมิร์ซลองทำนายการเลิกซื้อ (1 = เลิกซื้อ, 0 = ยังซื้ออยู่) ด้วย linear regression ธรรมดา (OLS โดยใส่ตัวแปรตามเป็น 0/1 ตรงๆ ไม่ใช่ logistic) จากระยะเวลาที่เป็นลูกค้า, จำนวนครั้งที่เข้าเว็บ/แอป, การเป็นสมาชิกสะสมแต้ม และอายุ ด้วยลูกค้าทั้ง 1203 คนในไฟล์ แล้วพบว่าบางคนได้ค่าทำนายที่ใช้เป็นความน่าจะเป็นไม่ได้ **มีลูกค้ากี่คนที่ค่าทำนายจาก OLS นี้ต่ำกว่า 0 หรือสูงกว่า 1** = ? คน (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit Linear Probability Model (OLS ธรรมดา)
cust = pd.read_csv('biz_customers.csv')
formula = 'churn ~ tenure_months + web_visits + loyalty_member + age'
ols_model = smf.ols(formula, data=cust).fit()

# 2. ทำนายค่า y_hat
preds = ols_model.predict(cust)
print(f"ค่าทำนายต่ำสุด: {preds.min():.4f}, ค่าทำนายสูงสุด: {preds.max():.4f}")

# 3. นับจำนวนแถวที่ค่าทำนาย < 0 หรือ > 1
invalid_count = int(np.sum((preds < 0) | (preds > 1)))
print(f"จำนวนลูกค้าที่ได้ความน่าจะเป็นผิดช่วง [0, 1]: {invalid_count} คน")
print(f"คำตอบข้อ 11.2: {invalid_count}")


ค่าทำนายต่ำสุด: -0.2456, ค่าทำนายสูงสุด: 0.3270
จำนวนลูกค้าที่ได้ความน่าจะเป็นผิดช่วง [0, 1]: 126 คน
คำตอบข้อ 11.2: 126


**โจทย์ข้อ 11.3:** ฝึกใช้ไฟล์ `biz_customers.csv` ตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `income_k`, `age` ประมาณด้วย Maximum Likelihood (ไม่มี regularization) fit โมเดลด้วยข้อมูล train (960 แถวแรกของไฟล์) แล้วทำนายลูกค้าในชุด test (243 แถวที่เหลือ) โดยจัดเป็น responded = 1 เมื่อความน่าจะเป็นที่ทำนาย $\ge 0.25$ **ค่า F1 score ของชุด test (คลาสบวก = 1)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import f1_score

# 1. แบ่งชุด train (960 แถวแรก) และ test (243 แถวที่เหลือ)
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดล Logit บนชุด train
formula = 'responded ~ email_opens + income_k + age'
model = smf.logit(formula, data=train).fit()

# 3. ทำนายบนชุด test ที่ threshold = 0.25
test_prob = model.predict(test)
test_pred = (test_prob >= 0.25).astype(int)

# 4. คำนวณ F1 score
f1 = f1_score(test['responded'], test_pred)
print(f"F1 score บนชุด test: {f1:.4f}")
print(f"คำตอบข้อ 11.3: {f1:.4f}")


Optimization terminated successfully.
         Current function value: 0.588269
         Iterations 5
F1 score บนชุด test: 0.5739
คำตอบข้อ 11.3: 0.5739


**โจทย์ข้อ 11.4:** กรณีศึกษาธนาคารแห่งหนึ่งสร้างโมเดล logistic regression ทำนายการตอบรับโปรโมชันบัตรเครดิตของลูกค้า จากจำนวนอีเมลโปรโมชันที่เปิดอ่าน, จำนวนครั้งที่เข้าเว็บ/แอป และรายได้ ด้วยลูกค้าทั้ง 1203 คนในไฟล์ (Maximum Likelihood ไม่มี regularization) ถ้าจำนวนอีเมลโปรโมชันที่เปิดอ่านเพิ่มขึ้น 3 ฉบับ โดยตัวแปรอื่นคงที่ **odds ของการตอบรับจะเพิ่มขึ้นเป็นกี่เท่า (odds ratio)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล Logit
cust = pd.read_csv('biz_customers.csv')
formula = 'responded ~ email_opens + web_visits + income_k'
model = smf.logit(formula, data=cust).fit()

# 2. ดึงสัมประสิทธิ์ของ email_opens และคำนวณ Odds Ratio เมื่อเพิ่ม 3 ฉบับ
beta_email = model.params['email_opens']
odds_ratio = np.exp(3.0 * beta_email)
print(f"beta ของ email_opens: {beta_email:.4f}")
print(f"Odds ratio สำหรับ 3 ฉบับ = exp(3 * beta): {odds_ratio:.4f}")
print(f"คำตอบข้อ 11.4: {odds_ratio:.4f}")


Optimization terminated successfully.
         Current function value: 0.578616
         Iterations 5
beta ของ email_opens: 0.1755
Odds ratio สำหรับ 3 ฉบับ = exp(3 * beta): 1.6930
คำตอบข้อ 11.4: 1.6930


**โจทย์ข้อ 11.5:** กรณีศึกษาร้านกาแฟสาขาหนึ่งต้องการรู้ว่าลูกค้าที่มีรายได้ 40 พันบาท/เดือนต้องเปิดอ่านอีเมลโปรโมชันกี่ฉบับ (ถือเป็นค่าต่อเนื่องได้) จึงจะมีความน่าจะเป็นที่จะตอบรับโปรโมชันเท่ากับ 0.4 ทีมงาน fit logistic regression ทำนายการตอบรับโปรโมชันจากจำนวนอีเมลโปรโมชันที่เปิดอ่านและรายได้ ด้วยลูกค้าทั้ง 1203 คนในไฟล์ (Maximum Likelihood ไม่มี regularization) **จำนวนอีเมลที่เปิดอ่านที่ได้** = ? ฉบับ (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล Logit
cust = pd.read_csv('biz_customers.csv')
formula = 'responded ~ email_opens + income_k'
model = smf.logit(formula, data=cust).fit()

# 2. ตั้งสมการ logit(p) = b0 + b_email * email + b_income * income
# ที่ p = 0.4 -> logit(0.4) = ln(0.4 / 0.6)
b0 = model.params['Intercept']
b_email = model.params['email_opens']
b_income = model.params['income_k']
target_logit = np.log(0.4 / 0.6)

# 3. แก้สมการหา email_opens เมื่อ income_k = 40
email_needed = (target_logit - b0 - b_income * 40.0) / b_email
print(f"จำนวนอีเมลที่ต้องเปิดอ่าน: {email_needed:.2f} ฉบับ")
print(f"คำตอบข้อ 11.5: {email_needed:.2f}")


Optimization terminated successfully.
         Current function value: 0.595553
         Iterations 5
จำนวนอีเมลที่ต้องเปิดอ่าน: 7.30 ฉบับ
คำตอบข้อ 11.5: 7.30


**โจทย์ข้อ 11.6:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `web_visits`, `age` (Maximum Likelihood ไม่มี regularization) fit ด้วยข้อมูล train (960 แถวแรกของ `biz_customers.csv`) ทำนายชุด test (243 แถวที่เหลือ) โดยจัดเป็นคลาส 1 เมื่อความน่าจะเป็นที่ทำนาย $\ge 0.4$ **จำนวนลูกค้าในชุด test ที่ตอบรับจริง (responded = 1) และโมเดลทำนายเป็นคลาส 1 ด้วย (True Positive)** = ? คน (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดล Logit บน train
formula = 'responded ~ email_opens + web_visits + age'
model = smf.logit(formula, data=train).fit()

# 3. ทำนายชุด test ที่ threshold = 0.4
test_prob = model.predict(test)
test_pred = (test_prob >= 0.4).astype(int)

# 4. หาค่า True Positives (TP)
tn, fp, fn, tp = confusion_matrix(test['responded'], test_pred).ravel()
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print(f"จำนวน True Positives: {tp} คน")
print(f"คำตอบข้อ 11.6: {tp}")


Optimization terminated successfully.
         Current function value: 0.581063
         Iterations 5
TN: 146, FP: 18, FN: 54, TP: 25
จำนวน True Positives: 25 คน
คำตอบข้อ 11.6: 25


**โจทย์ข้อ 11.7:** กรณีศึกษาธุรกิจฟิตเนสสมัครสมาชิกสร้างโมเดล logistic regression ทำนายการเลิกซื้อของลูกค้าจากระยะเวลาที่เป็นลูกค้า, จำนวนครั้งที่เข้าเว็บ/แอป, การเป็นสมาชิกสะสมแต้ม และอายุ (Maximum Likelihood ไม่มี regularization) ด้วยข้อมูล train (960 แถวแรก) แล้วนำไปคัดลูกค้าชุด test (243 แถวที่เหลือ) ทีมจะโทรเสนอสิทธิพิเศษให้ทุกคนที่ความน่าจะเป็นที่ทำนายว่าจะเลิกซื้อ $\ge 0.12$ **ในชุด test มีลูกค้าที่ต้องโทรทั้งหมดกี่คน** = ? คน (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดลทำนาย churn บน train
formula = 'churn ~ tenure_months + web_visits + loyalty_member + age'
model = smf.logit(formula, data=train).fit()

# 3. ทำนายความน่าจะเป็นบน test และนับจำนวนที่ prob >= 0.12
test_prob = model.predict(test)
count_to_call = int(np.sum(test_prob >= 0.12))
print(f"จำนวนลูกค้าที่ต้องโทรเสนอสิทธิพิเศษ: {count_to_call} คน")
print(f"คำตอบข้อ 11.7: {count_to_call}")


Optimization terminated successfully.
         Current function value: 0.314666
         Iterations 7
จำนวนลูกค้าที่ต้องโทรเสนอสิทธิพิเศษ: 88 คน
คำตอบข้อ 11.7: 88


**โจทย์ข้อ 11.8:** กรณีศึกษาผู้ให้บริการอินเทอร์เน็ตรายเดือนพบว่าลูกค้าที่เลิกซื้อในข้อมูล train (960 แถวแรก) มีน้อย จึงถ่วงน้ำหนักตัวอย่างแบบ `class_weight='balanced'` (น้ำหนักคำนวณจาก train) แล้ว fit โมเดลทำนายการเลิกซื้อจากระยะเวลาที่เป็นลูกค้า, จำนวนครั้งที่เข้าเว็บ/แอป และอายุ ด้วย Maximum Likelihood แบบถ่วงน้ำหนัก (ไม่มี regularization — ถ้าใช้ scikit-learn ต้องปิด penalty หรือใช้ `sm.GLM(..., family=sm.families.Binomial(), freq_weights=...)`) แล้วทำนายลูกค้าชุด test (243 แถวที่เหลือ) โดยจัดเป็นคลาส 1 เมื่อความน่าจะเป็นที่ทำนาย $\ge 0.5$ **ค่า recall ของชุด test (คลาสบวก = 1 คือเลิกซื้อ)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]
features = ['tenure_months', 'web_visits', 'age']

# 2. Fit โมเดลแบบ balanced class weights (ไม่มี regularization)
clf = LogisticRegression(penalty=None, class_weight='balanced')
clf.fit(train[features], train['churn'])

# 3. ทำนายชุด test ที่ threshold = 0.5 และคำนวณ Recall
test_prob = clf.predict_proba(test[features])[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
recall = recall_score(test['churn'], test_pred)
print(f"Recall บนชุด test: {recall:.4f}")
print(f"คำตอบข้อ 11.8: {recall:.4f}")


Recall บนชุด test: 0.5714
คำตอบข้อ 11.8: 0.5714


**โจทย์ข้อ 11.9:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `web_visits`, `income_k`, `age` ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ประมาณด้วย Maximum Likelihood (ไม่มี regularization) **ความน่าจะเป็นที่โมเดลทำนายว่าลูกค้าแถวที่ 562 (นับจาก 0) ตอบรับโปรโมชัน** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล Logit บนลูกค้าทั้งหมด 1203 คน
cust = pd.read_csv('biz_customers.csv')
formula = 'responded ~ email_opens + web_visits + income_k + age'
model = smf.logit(formula, data=cust).fit()

# 2. ทำนายความน่าจะเป็นของแถวที่ 562 (0-indexed)
prob_all = model.predict(cust)
prob_row562 = prob_all.iloc[562]
print(f"ข้อมูลลูกค้าแถว 562:\n{cust.iloc[562][['email_opens', 'web_visits', 'income_k', 'age']]}")
print(f"ความน่าจะเป็นที่จะตอบรับ: {prob_row562:.4f}")
print(f"คำตอบข้อ 11.9: {prob_row562:.4f}")


Optimization terminated successfully.
         Current function value: 0.576266
         Iterations 5
ข้อมูลลูกค้าแถว 562:
email_opens       4
web_visits        4
income_k       21.6
age              48
Name: 562, dtype: object
ความน่าจะเป็นที่จะตอบรับ: 0.1508
คำตอบข้อ 11.9: 0.1508


**โจทย์ข้อ 11.10:** ฝึกตัวแปรตาม = `responded` ตัวทำนาย = `email_opens`, `web_visits`, `income_k`, `age` (Maximum Likelihood ไม่มี regularization) fit ด้วยข้อมูล train (960 แถวแรกของ `biz_customers.csv`) แล้วลอง **threshold = 0.20, 0.25, 0.30, ..., 0.50 (ทั้งหมด 7 ค่า)** กับชุด test (243 แถวที่เหลือ) **ค่า F1 score สูงสุดในบรรดา 7 threshold นี้ (คลาสบวก = 1)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import f1_score

# 1. แบ่งชุด train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดล Logit บน train
formula = 'responded ~ email_opens + web_visits + income_k + age'
model = smf.logit(formula, data=train).fit()

# 3. ทดสอบทั้ง 7 Thresholds บนชุด test
test_prob = model.predict(test)
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

best_f1 = -1.0
best_th = None
for th in thresholds:
    pred_th = (test_prob >= th).astype(int)
    f1 = f1_score(test['responded'], pred_th)
    print(f"Threshold {th:.2f}: F1 score = {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_th = th

print(f"\nค่า F1 score สูงสุด: {best_f1:.4f} (ที่ threshold = {best_th:.2f})")
print(f"คำตอบข้อ 11.10: {best_f1:.4f}")


Optimization terminated successfully.
         Current function value: 0.579029
         Iterations 5
Threshold 0.20: F1 score = 0.5481
Threshold 0.25: F1 score = 0.5852
Threshold 0.30: F1 score = 0.5641
Threshold 0.35: F1 score = 0.5369
Threshold 0.40: F1 score = 0.3636
Threshold 0.45: F1 score = 0.2500
Threshold 0.50: F1 score = 0.1915

ค่า F1 score สูงสุด: 0.5852 (ที่ threshold = 0.25)
คำตอบข้อ 11.10: 0.5852


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 11

---

### 1. ข้อจำกัดของ Linear Probability Model (LPM) และแบบจำลอง Logistic Regression

- **Linear Probability Model (LPM):** เมื่อใช้ OLS ทำนายตัวแปรตามที่เป็น 0/1 ค่าทำนาย $\hat{y}$ อาจหลุดออกนอกช่วง $[0, 1]$ (เช่น ค่าติดลบหรือเกิน 1) และ Error term มีปัญหา Heteroscedasticity
- **Logistic Function:** แปลงค่าผลรวมเชิงเส้น $z = \beta_0 + \sum \beta_j X_j$ ให้อยู่ในช่วง $[0, 1]$ เสมอ:
  $$p(X) = \frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$$

---

### 2. Logit, Odds และ Odds Ratio

- **Odds (อัตราส่วนความน่าจะเป็น):**
  $$\text{Odds} = \frac{p}{1 - p}$$
- **Log-odds (Logit):**
  $$\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$$
- **Odds Ratio (OR):** เมื่อตัวแปร $X_j$ เพิ่มขึ้น $\Delta$ หน่วย:
  $$\text{OR} = \frac{\text{Odds}(X_j + \Delta)}{\text{Odds}(X_j)} = e^{\Delta \cdot \beta_j}$$

---

### 3. ตารางความสับสน (Confusion Matrix) และตัวชี้วัดประสิทธิภาพ

| ค่าจริง \ ทำนาย | $\hat{Y} = 1$ | $\hat{Y} = 0$ |
| :---: | :---: | :---: |
| **$Y = 1$** | **True Positive (TP)** | False Negative (FN) |
| **$Y = 0$** | False Positive (FP) | **True Negative (TN)** |

- **Precision:** $\frac{\text{TP}}{\text{TP} + \text{FP}}$ (ความแม่นยำของกลุ่มที่ทำนายว่าเป็นบวก)
- **Recall (Sensitivity):** $\frac{\text{TP}}{\text{TP} + \text{FN}}$ (สัดส่วนของคลาสจริงที่จับได้ครบ)
- **F1 Score:** ฮาร์มอนิกของ Precision และ Recall:
  $$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

---

### 4. การปรับ Decision Threshold และ Class Weighting

- **Threshold Tuning:** ในงานธุรกิจที่มี Class Imbalance หรือความสำคัญของ False Positive กับ False Negative ต่างกัน การลดเกณฑ์ (เช่น $\text{threshold} = 0.25$) จะช่วยเพิ่ม Recall และมักให้ F1 score สูงกว่าเกณฑ์ปกติ $0.50$
- **Balanced Class Weights:** คำนวณน้ำหนักตัวอย่างแบบผกผันกับความถี่คลาส:
  $$w_c = \frac{n}{2 \cdot n_c}$$
  เพื่อบังคับให้โมเดลให้ความสำคัญกับคลาสส่วนน้อย (Minority class เช่น Churn) มากขึ้น